# Loadsmart Data Challenge — Python Functions & Data Export

This notebook covers four topics:

| # | Topic |
|---|-------|
| 1 | `split_lane` — parse a lane string into pickup/delivery city and state |
| 2 | `send_csv_via_email` — attach and send a CSV file by SMTP email |
| 3 | `send_csv_via_sftp` — upload a CSV file to a remote server over sFTP |
| 4 | **Export** — query the dimensional model in BigQuery and produce the last-month delivery report |

---

In [72]:
# Install dependencies inside the notebook
%pip install google-cloud-bigquery google-cloud-bigquery-storage db-dtypes pandas paramiko pyarrow

Note: you may need to restart the kernel to use updated packages.


In [73]:
import os
import re
import smtplib
from email import encoders
from email.mime.base import MIMEBase
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from pathlib import Path

import pandas as pd
import paramiko
from google.cloud import bigquery
from google.oauth2 import service_account

---
## 1. `split_lane` — Parse lane into city / state components

The `lane` column contains strings like `"Hood River,OR -> Upper Marlboro,MD"`.  
This function extracts four fields: `pickup_city`, `pickup_state`, `delivery_city`, `delivery_state`.

In [74]:
def split_lane(lane: str) -> dict:
    pattern = r'^(.+),([A-Z]{2})\s*->\s*(.+),([A-Z]{2})$'
    match = re.match(pattern, lane.strip()) if isinstance(lane, str) else None
    if match:
        return {
            'pickup_city':    match.group(1).strip(),
            'pickup_state':   match.group(2).strip(),
            'delivery_city':  match.group(3).strip(),
            'delivery_state': match.group(4).strip(),
        }
    return {'pickup_city': None, 'pickup_state': None,
            'delivery_city': None, 'delivery_state': None}

In [75]:
# Quick tests
cases = [
    'Hood River,OR -> Upper Marlboro,MD',
    'Etowah,TN -> Reno,NV',
    'Los Angeles,CA -> Greensboro,NC',
]
for lane in cases:
    print(split_lane(lane))

{'pickup_city': 'Hood River', 'pickup_state': 'OR', 'delivery_city': 'Upper Marlboro', 'delivery_state': 'MD'}
{'pickup_city': 'Etowah', 'pickup_state': 'TN', 'delivery_city': 'Reno', 'delivery_state': 'NV'}
{'pickup_city': 'Los Angeles', 'pickup_state': 'CA', 'delivery_city': 'Greensboro', 'delivery_state': 'NC'}


In [76]:
# Apply to the raw seed CSV to verify at scale
CSV_PATH = Path('..') / 'seeds' / '2026_data_challenge_ae_data.csv'

raw = pd.read_csv(CSV_PATH)
lane_split = raw['lane'].apply(split_lane).apply(pd.Series)
df_with_lanes = pd.concat([raw[['loadsmart_id', 'lane']], lane_split], axis=1)

print(f'Total rows : {len(df_with_lanes)}')
print(f'Parse failures: {lane_split["pickup_city"].isna().sum()}')
df_with_lanes.head()

Total rows : 5361
Parse failures: 0


,loadsmart_id,lane,pickup_city,pickup_state,delivery_city,delivery_state
0,206431033,"Hood River,OR -> Upper Marlboro,MD",Hood River,OR,Upper Marlboro,MD
1,206521177,"Etowah,TN -> Reno,NV",Etowah,TN,Reno,NV
2,206694049,"Salinas,CA -> Upper Marlboro,MD",Salinas,CA,Upper Marlboro,MD
3,206553113,"Montpelier,OH -> Reno,NV",Montpelier,OH,Reno,NV
4,206518817,"Newark,DE -> Portland,OR",Newark,DE,Portland,OR


---
## 2. `send_csv_via_email` — Send a CSV file as an email attachment

Uses Python's built-in `smtplib` with STARTTLS. Compatible with Gmail, Outlook, or any SMTP server.

In [77]:
def send_csv_via_email(
    csv_file_path: str,
    subject: str,
    body: str,
    to_email: str,
    from_email: str,
    smtp_host: str = 'smtp.gmail.com',
    smtp_port: int = 587,
    smtp_user: str = None,
    smtp_password: str = None,
) -> None:
    """
    Send a CSV file as an email attachment via SMTP/STARTTLS.

    Parameters
    ----------
    csv_file_path : str   Path to the local CSV file.
    subject       : str   Email subject line.
    body          : str   Plain-text email body.
    to_email      : str   Recipient address.
    from_email    : str   Sender address.
    smtp_host     : str   SMTP server hostname (default: Gmail).
    smtp_port     : int   SMTP port (default: 587 for STARTTLS).
    smtp_user     : str   Login username; defaults to from_email.
    smtp_password : str   SMTP password or app password.
    """
    csv_path = Path(csv_file_path)
    if not csv_path.exists():
        raise FileNotFoundError(f'CSV not found: {csv_file_path}')

    msg = MIMEMultipart()
    msg['From']    = from_email
    msg['To']      = to_email
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))

    with open(csv_path, 'rb') as f:
        part = MIMEBase('application', 'octet-stream')
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename="{csv_path.name}"')
    msg.attach(part)

    with smtplib.SMTP(smtp_host, smtp_port) as server:
        server.ehlo()
        server.starttls()
        server.login(smtp_user or from_email, smtp_password)
        server.sendmail(from_email, to_email, msg.as_string())

    print(f"Email sent to {to_email} | attachment: {csv_path.name}")

In [78]:
# Usage example (fill in real credentials before running)
# You need to change the SMTP_PASSWORD to the APP Password if you'll use gmail
import os                                                                                                                                                                                                                                                                                                  

os.environ['SMTP_PASSWORD'] = 'lzjw tlnj qkak lyxj'
print(repr(os.getenv('SMTP_PASSWORD')))   

'lzjw tlnj qkak lyxj'


In [79]:
# Usage example (fill in real credentials before running)
# I used my own email to test

send_csv_via_email(
    csv_file_path = '/Users/pam_v/Library/CloudStorage/OneDrive-Personal/Documentos/Projetos/loadsmart/loadsmart_dbt/seeds/2026_data_challenge_ae_data.csv',
    subject       = 'Deliveries Report',
    body          = 'Please find the delivery report attached.',
    to_email      = 'pamvnunes@gmail.com',
    from_email    = 'pamvnunes@gmail.com',
    smtp_password = os.getenv('SMTP_PASSWORD'),
)

Email sent to pamvnunes@gmail.com | attachment: 2026_data_challenge_ae_data.csv


---
## 3. `send_csv_via_sftp` — Upload a CSV file over sFTP

Uses `paramiko`. Supports both password and private-key authentication.

In [80]:
def send_csv_via_sftp(
    csv_file_path: str,
    destination_file_path: str,
    hostname: str,
    username: str,
    password: str = None,
    private_key_path: str = None,
    port: int = 22,
) -> None:
    """
    Upload a CSV file to a remote server over sFTP.

    Parameters
    ----------
    csv_file_path         : str   Local path to the CSV file.
    destination_file_path : str   Full remote path including filename.
    hostname              : str   sFTP server hostname or IP.
    username              : str   SSH/sFTP username.
    password              : str   Password (use None when using a private key).
    private_key_path      : str   Path to RSA/Ed25519 private key file.
    port                  : int   SSH port (default: 22).
    """
    csv_path = Path(csv_file_path)
    if not csv_path.exists():
        raise FileNotFoundError(f'CSV not found: {csv_file_path}')

    transport = paramiko.Transport((hostname, port))
    try:
        if private_key_path:
            pkey = paramiko.RSAKey.from_private_key_file(private_key_path)
            transport.connect(username=username, pkey=pkey)
        else:
            transport.connect(username=username, password=password)

        sftp = paramiko.SFTPClient.from_transport(transport)
        try:
            sftp.put(str(csv_path), destination_file_path)
            print(f'Uploaded {csv_path.name} → {hostname}:{destination_file_path}')
        finally:
            sftp.close()
    finally:
        transport.close()

In [81]:
# Usage example (fill in real server details before running)
# I used my own notebook to test
# If you are gonna use a SFTP to test, change the password for private_key_path

send_csv_via_sftp(
    csv_file_path = '/Users/pam_v/Library/CloudStorage/OneDrive-Personal/Documentos/Projetos/loadsmart/loadsmart_dbt/seeds/2026_data_challenge_ae_data.csv',
    destination_file_path = '/Users/pam_v/Library/CloudStorage/OneDrive-Personal/Documentos/Projetos/loadsmart/loadsmart_dbt/2026_data_challenge_ae_data.csv',
    hostname = 'localhost',
    username = 'pam_v',
    password = '1113', # Comment this line to use the private_key_path
    # private_key_path = 'YOUR_KEY'
)                                                                                                                                                                                                                                                   

Uploaded 2026_data_challenge_ae_data.csv → localhost:/Users/pam_v/Library/CloudStorage/OneDrive-Personal/Documentos/Projetos/loadsmart/loadsmart_dbt/2026_data_challenge_ae_data.csv


---
## 4. Export — Last Month Deliveries from the Dimensional Model

Connects to BigQuery, joins `fct_loads` with the dimension tables on `loadsmart_id`, and exports  
all non-cancelled loads delivered in the **last available calendar month** to a CSV file.

**Output columns:** `loadsmart_id`, `shipper_name`, `delivery_dt`, `pickup_city`,  
`pickup_state`, `delivery_city`, `delivery_state`, `book_price_nm`, `carrier_name`

In [82]:
# BigQuery configuration
# You need to change the enviroments configurations

BQ_PROJECT = os.getenv('DBT_PROJECT_ID', 'loadsmart-challenge-494016')
BQ_DATASET = os.getenv('DBT_DATASET', 'analytics')
BQ_KEYFILE = os.getenv('DBT_KEYFILE', '/Users/pam_v/Downloads/loadsmart-challenge-494016-58245994946a.json')


def get_bq_client() -> bigquery.Client:
    """Return a BigQuery client using a service-account keyfile or ADC."""
    if BQ_KEYFILE and Path(BQ_KEYFILE).exists():
        credentials = service_account.Credentials.from_service_account_file(BQ_KEYFILE)
        return bigquery.Client(project=BQ_PROJECT, credentials=credentials)
    # Falls back to Application Default Credentials (gcloud auth)
    return bigquery.Client(project=BQ_PROJECT)

In [ ]:
def export_last_month_deliveries(
    output_path: str = 'last_month_deliveries.csv',
) -> pd.DataFrame:
    """
    Query the dimensional model for loads delivered in the last available
    calendar month and export the result to a CSV file.

    The 'last available month' is derived from MAX(delivery_dt) in fct_loads,
    excluding cancelled loads.

    Parameters
    ----------
    output_path : str  Destination path for the CSV file.

    Returns
    -------
    pd.DataFrame  The exported data.
    """
    client = get_bq_client()

    query = f"""
    WITH last_month AS (
        SELECT DATE_TRUNC(MAX(delivery_dt), MONTH) AS period_start
        FROM `{BQ_PROJECT}.{BQ_DATASET}.fct_loads`
        WHERE NOT was_cancelled
    )
    SELECT
        f.loadsmart_id,
        ds.shipper_nm,
        f.delivery_dt,
        dl.origin_city       AS pickup_city,
        dl.origin_state      AS pickup_state,
        dl.destination_city  AS delivery_city,
        dl.destination_state AS delivery_state,
        f.book_price_nm,
        dc.carrier_nm
    FROM `{BQ_PROJECT}.{BQ_DATASET}.fct_loads` f
    LEFT JOIN `{BQ_PROJECT}.{BQ_DATASET}.dim_shipper` ds ON f.loadsmart_id = ds.loadsmart_id
    LEFT JOIN `{BQ_PROJECT}.{BQ_DATASET}.dim_lane` dl ON f.loadsmart_id = dl.loadsmart_id
    LEFT JOIN `{BQ_PROJECT}.{BQ_DATASET}.dim_carrier` dc ON f.loadsmart_id = dc.loadsmart_id
    CROSS JOIN last_month lm
    WHERE DATE_TRUNC(f.delivery_dt, MONTH) = lm.period_start
      AND NOT f.was_cancelled
    ORDER BY f.delivery_dt, f.loadsmart_id
    """

    df = client.query(query).to_dataframe()
    df.to_csv(output_path, index=False)
    print(f'Exported {len(df)} rows → {output_path}')
    return df

In [84]:
# Run the export (requires BigQuery access and dbt models to be built)
df_deliveries = export_last_month_deliveries('last_month_deliveries.csv')
df_deliveries.head(10)

Exported 1 rows → last_month_deliveries.csv


,loadsmart_id,shipper_name,delivery_dt,pickup_city,pickup_state,delivery_city,delivery_state,book_price_nm,carrier_name
0,206665369,Shipper 758,2025-03-15,Lodi,CA,Pacific,WA,1955.560000000,Carrier 86454


In [ ]:
# Quick summary
print('Delivery month :', df_deliveries['delivery_dt'].astype(str).str[:7].unique())
print('Total loads    :', len(df_deliveries))
print('Shippers       :', df_deliveries['shipper_nm'].nunique())
print('Carriers       :', df_deliveries['carrier_nm'].nunique())
print('Total revenue  :', df_deliveries['book_price_nm'].sum().round(2))

Delivery month : ['2025-03']
Total loads    : 1
Shippers       : 1
Carriers       : 1


AttributeError: 'decimal.Decimal' object has no attribute 'round'